# Multi-Source Entity Resolution Pipeline (v2 — GPU/disk-optimized)

**Task:** Source 1 is a deduplicated reference set of business entities. For every Source 1
record we must find all matching records in Source 2 and Source 3 (zero, one, or many matches
each). The test set additionally contains a third country (France) that never appears during
training, so nothing below hard-codes or filters on a fixed country list — `country` is treated
as an open set of strings throughout.

This version reworks the pipeline around **hardware budget** (≤30GB RAM / ≤16GB GPU / ≤150GB
disk) and a **tighter, simpler candidate-generation design**:

| # | Change vs. the previous notebook |
|---|---|
| 1 | Embeddings are computed **batch-wise** (`EMBEDDING_BATCH_SIZE` texts per `.encode()` call) — peak RAM/GPU is one batch, never the whole source |
| 2 | Embeddings are **memory-mapped on disk** (`.npy` memmap) the moment they're computed — never held as one big in-RAM array |
| 3 | FAISS index building and FAISS search both run **on GPU**, and every retrieval result is **flushed to a CSV on disk immediately**; the modelling stage re-loads candidates *from disk* rather than continuing to hold the in-memory object from the retrieval cell |
| 4 | FAISS similarity is computed on the **combined `name + address` embedding only** |
| 5 | The separate **name-only and address-only embeddings are removed entirely** (name/address string similarity is instead captured later by RapidFuzz + Jaccard features, which is cheaper and doesn't need a second embedding model pass) |
| 6 | Candidate generation is exactly **country block → top-40 by FAISS score** — no exact-name retrieval channel, no global fallback (see the Step 4 note for why this is safe) |
| 7 | The matching model's features are **RapidFuzz string-similarity + Jaccard token-overlap** (+ the FAISS score/rank + exact-match/normalization flags) |
| 8 | Label creation is followed by an explicit **label-distribution check** (counts, positive rate, positive rate by source/country, and a plot) |
| 9 | The classifier is tuned by sweeping the decision threshold to maximize **F0.5 on the match class** |
| 10 | The pipeline writes **`candidates.csv`** (every surviving candidate pair + all features + model probability) and **`matching_pairs.csv`** (only the pairs the model predicts as matches) as first-class outputs, in addition to the official `submission.tsv` |

Ten stages, one-to-one with the changes above:

| Stage | What happens |
|---|---|
| 1 | Load raw TSVs, normalize & tokenize names/addresses/countries |
| 2 | Batch-wise combined-text embeddings, written straight to disk-backed `.npy` memmaps |
| 3 | Country-wise blocking + FAISS-GPU index building (combined embedding only) |
| 4 | Country-blocked top-40 FAISS retrieval, streamed to disk as it's computed |
| 5 | Dedup + disk checkpoint (candidates are reloaded from disk before feature engineering) |
| 6 | Feature engineering: FAISS score/rank + RapidFuzz + Jaccard + exact-match flags |
| 7 | Label creation from ground truth + label-distribution check |
| 8 | Group-based train/validation split + LightGBM training |
| 9 | Threshold tuning to maximize F0.5 on the match class |
| 10 | Inference on the test set (France included), `candidates.csv`, `matching_pairs.csv`, and `submission.tsv` |

> **Note on Step 1:** the raw files only contain `entity_id`, `business_name`, `business_address`,
> `country` (source is inferred from the `entity_id` prefix `S1-` / `S2-` / `S3-`). Step 1
> performs the normalization/tokenization the rest of the pipeline expects.

Every helper function below is written once and reused for train **and** test data, so Step 10
does not duplicate any logic from Steps 1–6 — it just calls the same functions again.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np  # linear algebra
import pandas as pd  # data processing, TSV file I/O

# Input data files are available in the read-only "../input/" directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to /kaggle/working/ that gets preserved when you "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside the session

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## 0. Environment Setup

Install dependencies. If you have a CUDA GPU available, prefer `faiss-gpu` (or the newer
`faiss-gpu-cu12` wheel for CUDA 12) over `faiss-cpu` — Steps 3–4 run their index build and
search **on GPU**. The code below auto-detects GPU support and gracefully falls back to CPU
indices if a GPU isn't available, so the notebook runs either way (just slower on Step 3/4).

In [2]:
# !pip install -q pandas numpy tqdm sentence-transformers torch rapidfuzz lightgbm scikit-learn joblib matplotlib
# !pip install -q faiss-cpu          # CPU-only FAISS (safe default)
# !pip install -q faiss-gpu            # OR: use this instead if you have a CUDA GPU + matching CUDA toolkit

In [3]:
import os
import gc
import json
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from sentence_transformers import SentenceTransformer

import faiss

from rapidfuzz import fuzz

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import fbeta_score, precision_score, recall_score, classification_report

import lightgbm as lgb
import joblib

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 0.1 Configuration

Central place for every path / hyperparameter used later. **Edit `DATA_DIR`** (and file names,
if yours differ) to point at your actual files — nothing else in the notebook needs to change.

`EMBED_DIR` / `CAND_DIR` / `MODEL_DIR` are where the disk-backed artifacts from Steps 2–5 live —
embeddings as `.npy` memmaps, raw + deduped candidate pairs as `.csv` checkpoints, and the
trained model, respectively. All of these persist across kernel restarts (as long as
`ARTIFACT_DIR` itself does), so re-running the notebook after an interruption skips
recomputation wherever a cached file already exists on disk.

In [ ]:
# ----------------------------- Paths ---------------------------------------
DATA_DIR = "/dataset/train"   # <-- EDIT ME
ARTIFACT_DIR = "/dataset/working"

EMBED_DIR = f"{ARTIFACT_DIR}/embeddings"   # disk-backed .npy memmaps (Step 2)
CAND_DIR = f"{ARTIFACT_DIR}/candidates"    # raw + deduped candidate-pair CSV checkpoints (Steps 4-5)
MODEL_DIR = f"{ARTIFACT_DIR}/model"        # trained model artifact (Step 9)
for _d in (ARTIFACT_DIR, EMBED_DIR, CAND_DIR, MODEL_DIR):
    os.makedirs(_d, exist_ok=True)

TRAIN_S1_PATH = f"{DATA_DIR}/train_source1.tsv"
TRAIN_S2_PATH = f"{DATA_DIR}/train_source2.tsv"
TRAIN_S3_PATH = f"{DATA_DIR}/train_source3.tsv"
TRAIN_GT_PATH = f"{DATA_DIR}/train_ground_truth.tsv"

TEST_S1_PATH = f"{DATA_DIR}/test_source1.tsv"
TEST_S2_PATH = f"{DATA_DIR}/test_source2.tsv"
TEST_S3_PATH = f"{DATA_DIR}/test_source3.tsv"

SUBMISSION_PATH = f"{ARTIFACT_DIR}/submission.tsv"
CANDIDATES_CSV_PATH = f"{ARTIFACT_DIR}/candidates.csv"          # Step 10 deliverable
MATCHING_PAIRS_CSV_PATH = f"{ARTIFACT_DIR}/matching_pairs.csv"  # Step 10 deliverable

# ----------------------------- Embedding model ------------------------------
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
EMBEDDING_BATCH_SIZE = 256          # texts per .encode() call -> bounds peak GPU/RAM to one batch
EMBEDDING_STORAGE_DTYPE = "float16"  # halves on-disk size; upcast to float32 right before FAISS ops
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------- FAISS (Steps 3-4) ----------------------------
FAISS_ADD_BATCH = 20000     # rows added to a FAISS index per .add() call
FAISS_QUERY_BATCH = 4096    # S1 rows searched per .search() call
TOP_K_CANDIDATES = 40       # country-blocked top-K per S1 record (per source)

# ----------------------------- Training (Step 8-9) ---------------------------
VALIDATION_FRACTION = 0.2
LGBM_PARAMS = {
    "objective": "binary",
    "metric": "binary_logloss",
    "is_unbalance": True,          # candidate pairs are heavily imbalanced (few true matches)
    "learning_rate": 0.05,
    "num_leaves": 63,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "min_data_in_leaf": 20,
    "verbosity": -1,
    "seed": RANDOM_STATE,
}
LGBM_NUM_BOOST_ROUND = 2000
LGBM_EARLY_STOPPING_ROUNDS = 50

# ----------------------------- Submission (Step 10) --------------------------
TOP_K_PER_SOURCE_IN_SUBMISSION = 10   # "Top 10 from S2, Top 10 from S3 per S1"

print(f"Using device: {DEVICE}")
print(f"FAISS version: {faiss.__version__ if hasattr(faiss, '__version__') else 'unknown'}")

Using device: cuda
FAISS version: 1.15.1


## Step 1 — Load Data, Normalize & Tokenize

Builds, for every record:
- `name_norm`, `name_tokens` — lower-cased, punctuation-stripped business name + its tokens
- `name_suffix_norm` — name with legal suffixes mapped to a canonical short form (`Corporation`→`corp`)
- `name_nosuffix` — name with those suffix tokens removed entirely (handles Corp/Pvt/Ltd noise)
- `name_sorted` — alphabetically sorted tokens of `name_nosuffix` (handles word-order transpositions)
- `address_norm`, `address_tokens` — normalized address + tokens (common abbreviations expanded)
- `address_numeric_tokens` — just the tokens containing digits (PIN codes, house/door numbers, etc.)
- `country_norm` — standardized country string (open set — unknown labels are upper-cased and
  *kept*, never dropped, so France in the test set is handled automatically)
- `source` — 'S1' / 'S2' / 'S3', parsed from the `entity_id` prefix

These fields feed **feature engineering only** now (Step 6) — candidate generation itself
(Step 4) is FAISS-only, so the exact-match variants (`name_suffix_norm`, `name_nosuffix`,
`name_sorted`) are used as engineered similarity flags rather than as a separate retrieval
channel.

In [7]:
# Canonical forms for common legal-entity suffixes (extend this dict as you see more noise).
LEGAL_SUFFIXES = {
    "corporation": "corp", "corp": "corp",
    "incorporated": "inc", "inc": "inc",
    "limited": "ltd", "ltd": "ltd",
    "private": "pvt", "pvt": "pvt",
    "company": "co", "co": "co",
    "llc": "llc", "l.l.c": "llc",
    "llp": "llp", "plc": "plc",
    "pty": "pty", "gmbh": "gmbh",
    "enterprises": "ent", "ent": "ent",
    "industries": "inds", "inds": "inds",
}
SUFFIX_CANONICAL_VALUES = set(LEGAL_SUFFIXES.values())

# Common address-token abbreviations (extend as needed for your data).
ADDRESS_ABBREV = {
    "road": "rd", "rd": "rd",
    "street": "st", "st": "st",
    "avenue": "ave", "ave": "ave",
    "boulevard": "blvd", "blvd": "blvd",
    "lane": "ln", "ln": "ln",
    "drive": "dr", "dr": "dr",
    "apartment": "apt", "apt": "apt",
    "building": "bldg", "bldg": "bldg",
    "floor": "fl", "fl": "fl",
    "nagar": "nagar", "colony": "colony",
    "near": "near", "nr": "near",
    "sector": "sec", "sec": "sec",
}

# Known aliases -> standardized code. Anything NOT in this dict is simply upper-cased and kept
# as-is (open-set behaviour) -- this is what makes France "just work" at test time.
COUNTRY_ALIASES = {
    "usa": "US", "us": "US", "u.s.a": "US", "u.s": "US",
    "united states": "US", "united states of america": "US",
    "india": "IN", "in": "IN", "bharat": "IN",
    "france": "FR", "fr": "FR",
}

def tokenize(s):
    return s.split() if s else []

def vectorized_text_clean(series):
    # Vectorized (C-level) text cleaning.
    s = series.fillna("").astype(str).str.lower().str.strip()
    s = s.str.replace(r"[^\w\s&]", " ", regex=True)
    s = s.str.replace("&", " and ", regex=False)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s


def build_name_features(name_series):
    name_norm = vectorized_text_clean(name_series)
    tokens_series = name_norm.str.split()

    suffix_norm_list, nosuffix_list, sorted_list = [], [], []
    for tokens in tokens_series:
        mapped = [LEGAL_SUFFIXES.get(t, t) for t in tokens]
        suffix_norm_list.append(" ".join(mapped))
        nosuffix_tokens = [t for t in mapped if t not in SUFFIX_CANONICAL_VALUES]
        nosuffix_list.append(" ".join(nosuffix_tokens) if nosuffix_tokens else " ".join(mapped))
        sort_source = nosuffix_tokens if nosuffix_tokens else mapped
        sorted_list.append(" ".join(sorted(sort_source)))

    return pd.DataFrame({
        "name_norm": name_norm.values,
        "name_tokens": tokens_series.values,
        "name_suffix_norm": suffix_norm_list,
        "name_nosuffix": nosuffix_list,
        "name_sorted": sorted_list,
    }, index=name_series.index)


def build_address_features(address_series):
    address_norm_base = vectorized_text_clean(address_series)
    tokens_series = address_norm_base.str.split()

    mapped_list, numeric_list, joined = [], [], []
    for tokens in tokens_series:
        mapped = [ADDRESS_ABBREV.get(t, t) for t in tokens]
        mapped_list.append(mapped)
        numeric_list.append([t for t in tokens if any(ch.isdigit() for ch in t)])
        joined.append(" ".join(mapped))

    return pd.DataFrame({
        "address_norm": joined,
        "address_tokens": mapped_list,
        "address_numeric_tokens": numeric_list,
    }, index=address_series.index)


def vectorized_normalize_country(series):
    # country_norm becomes a `category` dtype downstream -- the bigger RAM win here since
    # country has very few unique values.
    s = series.fillna("UNKNOWN").astype(str).str.strip().str.lower()
    mapped = s.map(COUNTRY_ALIASES)
    return mapped.fillna(s.str.upper())

def load_source_file(path):
    # Load one raw source TSV and attach every normalized/tokenized field. Works identically
    # for S1, S2, S3, and for train or test files.
    df = pd.read_csv(path, sep="\t")
    df = df.reset_index(drop=True)

    name_feats = build_name_features(df["business_name"])
    addr_feats = build_address_features(df["business_address"])
    df = pd.concat([df, name_feats, addr_feats], axis=1)

    df["country_norm"] = vectorized_normalize_country(df["country"]).astype("category")
    df["source"] = df["entity_id"].str.extract(r"^(S\d)-", expand=False)

    return df

def load_ground_truth(path):
    return pd.read_csv(path, sep="\t", dtype=str)

In [8]:
# ----- Load TRAIN data -----
s1_train = load_source_file(TRAIN_S1_PATH).reset_index(drop=True)
s2_train = load_source_file(TRAIN_S2_PATH).reset_index(drop=True)
s3_train = load_source_file(TRAIN_S3_PATH).reset_index(drop=True)
gt_train = load_ground_truth(TRAIN_GT_PATH)

print(f"S1 train: {len(s1_train):,} rows | S2 train: {len(s2_train):,} rows | "
      f"S3 train: {len(s3_train):,} rows | ground truth rows: {len(gt_train):,}")
print("\nCountries seen in training data:",
      sorted(set(s1_train['country_norm']) | set(s2_train['country_norm']) | set(s3_train['country_norm'])))

display(s1_train.head(3))

S1 train: 2,206,821 rows | S2 train: 5,034,616 rows | S3 train: 5,285,603 rows | ground truth rows: 2,206,821

Countries seen in training data: ['IN', 'US']


,entity_id,business_name,business_address,country,name_norm,name_tokens,name_suffix_norm,name_nosuffix,name_sorted,address_norm,address_tokens,address_numeric_tokens,country_norm,source
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US,orelee s barbershop,"[orelee, s, barbershop]",orelee s barbershop,orelee s barbershop,barbershop orelee s,1795 westchester dr high point nc,"[1795, westchester, dr, high, point, nc]",[1795],US,S1
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US,prime money,"[prime, money]",prime money,prime money,money prime,17560 ellis rd tahlequah ok,"[17560, ellis, rd, tahlequah, ok]",[17560],US,S1
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US,b retail inc,"[b, retail, inc]",b retail inc,b retail,b retail,1712 montebello ave phoenix az,"[1712, montebello, ave, phoenix, az]",[1712],US,S1


In [9]:
# Sanity checks -- catch schema/normalization problems early.
for name, df in [("S1", s1_train), ("S2", s2_train), ("S3", s3_train)]:
    assert df["entity_id"].is_unique, f"{name}: duplicate entity_id found"
    assert df["source"].notna().all(), f"{name}: could not parse source prefix for some rows"
    n_empty_name = (df["name_norm"] == "").sum()
    n_empty_addr = (df["address_norm"] == "").sum()
    print(f"{name}: {n_empty_name} rows with empty normalized name, "
          f"{n_empty_addr} rows with empty normalized address")

S1: 0 rows with empty normalized name, 0 rows with empty normalized address
S2: 2 rows with empty normalized name, 168967 rows with empty normalized address
S3: 13 rows with empty normalized name, 175916 rows with empty normalized address


## Step 2 — Batch-Wise Embeddings, Written Straight to Disk

**One embedding per record now: the combined text**
`"business name: {name} business address: {address}"`, L2-normalized (so inner product ==
cosine similarity, which is what lets Step 3 use a plain `IndexFlatIP` FAISS index). The
separate name-only and address-only embeddings from the earlier version are gone — that
signal is now captured more cheaply by the RapidFuzz + Jaccard features in Step 6, so we no
longer pay for a second/third embedding pass.

**How this stays within a tight RAM/GPU budget:**
- `model.encode()` is called in chunks of `EMBEDDING_BATCH_SIZE` texts. Peak GPU memory is one
  batch's worth of forward passes, never the whole source.
- Each batch is written directly into a `.npy` memmap on disk as soon as it's computed
  (`np.lib.format.open_memmap`). The full embedding matrix is never materialized in RAM — only
  one batch is briefly in memory before being flushed to disk.
- Embeddings are stored as `float16` (halves on-disk size vs. `float32`); they're upcast to
  `float32` only for the small slice actually being fed into a FAISS `.add()`/`.search()` call
  in Steps 3–4, so RAM/GPU cost of that upcast is bounded to one chunk at a time.
- Re-running the notebook reuses the cached `.npy` file on disk instead of re-encoding.

In [10]:
print(f"Loading embedding model '{EMBEDDING_MODEL_NAME}' on {DEVICE} ...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=DEVICE)

Loading embedding model 'paraphrase-multilingual-MiniLM-L12-v2' on cuda ...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
def encode_to_disk(texts, model, out_path, batch_size=EMBEDDING_BATCH_SIZE,
                    storage_dtype=EMBEDDING_STORAGE_DTYPE):
    # Encode in batches, writing each batch directly into a disk-backed .npy memmap. Peak RAM
    # usage is one batch of embeddings, not the whole array.
    dim = model.get_sentence_embedding_dimension()
    n = len(texts)
    mmap_arr = np.lib.format.open_memmap(out_path, mode="w+", dtype=storage_dtype, shape=(n, dim))
    for start in tqdm(range(0, n, batch_size), desc=os.path.basename(out_path)):
        batch = texts[start:start + batch_size]
        emb = model.encode(batch, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        mmap_arr[start:start + len(batch)] = emb.astype(storage_dtype)
    mmap_arr.flush()
    del mmap_arr
    gc.collect()
    return np.load(out_path, mmap_mode="r")  # reopen read-only, memory-mapped -- not loaded into RAM


def compute_combined_embeddings(df, model, prefix):
    combined_texts = ("business name: " + df["name_norm"].fillna("") +
                       " business address: " + df["address_norm"].fillna("")).tolist()
    out_path = f"{EMBED_DIR}/{prefix}_combined.npy"
    return encode_to_disk(combined_texts, model, out_path)


def get_or_compute_combined_embeddings(df, prefix, model):
    out_path = f"{EMBED_DIR}/{prefix}_combined.npy"
    if os.path.exists(out_path):
        print(f"Loading cached, disk-mapped combined embedding for {prefix} ...")
        return np.load(out_path, mmap_mode="r")
    print(f"Encoding {prefix} ({len(df):,} records) directly to disk (batch_size={EMBEDDING_BATCH_SIZE}) ...")
    return compute_combined_embeddings(df, model, prefix)

In [ ]:
s1_train_emb = get_or_compute_combined_embeddings(s1_train, "train_s1", embedding_model)
s2_train_emb = get_or_compute_combined_embeddings(s2_train, "train_s2", embedding_model)
s3_train_emb = get_or_compute_combined_embeddings(s3_train, "train_s3", embedding_model)

print("Embedding shapes:", s1_train_emb.shape, s2_train_emb.shape, s3_train_emb.shape)
print("Embedding dtype on disk:", s1_train_emb.dtype)

Encoding train_s1 (2,206,821 records) directly to disk (batch_size=256) ...


train_s1_combined.npy:   0%|          | 0/8621 [00:00<?, ?it/s]

## Step 3 — Country-wise Blocking & FAISS-GPU Index Building

For **S2 and S3 separately**: group records by `country_norm`, then build **one** `IndexFlatIP`
FAISS index per country (combined embedding only — no more per-embedding-type indices).

GPU acceleration is used automatically when available (`faiss.StandardGpuResources`), with a
silent fallback to CPU indices otherwise. Vectors are added to each index in
`FAISS_ADD_BATCH`-sized chunks read straight from the disk-backed `.npy` memmap (and upcast
`float16 -> float32` only for that chunk), so building an index never requires materializing
the full per-country embedding block in RAM at once.

`country_norm` is treated as an open string set (`groupby(..., observed=True)`), so a country
block for France is created automatically at test time without any code change.

In [ ]:
def get_faiss_gpu_resources():
    # Return a FAISS GPU resource handle if a GPU build of faiss is available, else None.
    try:
        if DEVICE == "cuda" and hasattr(faiss, "StandardGpuResources"):
            return faiss.StandardGpuResources()
    except Exception as e:
        print(f"FAISS GPU resources unavailable, falling back to CPU indices ({e})")
    return None


FAISS_GPU_RES = get_faiss_gpu_resources()
print("FAISS GPU acceleration:", "ENABLED" if FAISS_GPU_RES is not None else "disabled (using CPU indices)")


def build_flat_ip_index_from_memmap(emb_memmap, row_positions, gpu_res=FAISS_GPU_RES,
                                     add_batch=FAISS_ADD_BATCH):
    # Build an IndexFlatIP (cosine similarity, since embeddings are L2-normalized), adding rows
    # in batches read from a disk-backed memmap so only one batch is upcast to float32 in
    # RAM/GPU at a time.
    dim = emb_memmap.shape[1]
    cpu_index = faiss.IndexFlatIP(dim)
    index = faiss.index_cpu_to_gpu(gpu_res, 0, cpu_index) if gpu_res is not None else cpu_index
    for start in range(0, len(row_positions), add_batch):
        chunk_pos = row_positions[start:start + add_batch]
        vecs = np.asarray(emb_memmap[chunk_pos], dtype="float32")
        index.add(vecs)
    return index


def build_country_indices(df, emb_memmap, country_col="country_norm", gpu_res=FAISS_GPU_RES):
    # Build one per-country FAISS index (combined embedding only).
    #
    # Returns
    # -------
    # country_indices : dict[country] -> (faiss_index, entity_id_array)
    #
    # `df` must have its default RangeIndex (0..n-1) aligned with the rows of `emb_memmap`.
    df = df.reset_index(drop=True)
    entity_ids = df["entity_id"].values

    country_indices = {}
    for country, group_idx in df.groupby(country_col, observed=True).indices.items():
        group_idx = np.asarray(group_idx)
        idx = build_flat_ip_index_from_memmap(emb_memmap, group_idx, gpu_res)
        country_indices[country] = (idx, entity_ids[group_idx])

    return country_indices

In [ ]:
s2_train = s2_train.reset_index(drop=True)
s3_train = s3_train.reset_index(drop=True)

s2_country_idx = build_country_indices(s2_train, s2_train_emb)
s3_country_idx = build_country_indices(s3_train, s3_train_emb)

print("S2 countries indexed:", sorted(s2_country_idx.keys()))
print("S3 countries indexed:", sorted(s3_country_idx.keys()))

## Step 4 — Candidate Generation: Country Block → Top-40 by FAISS Score

For each Source-1 record, using **its own `country_norm`** to select the matching per-country
FAISS index built in Step 3, retrieve the top `TOP_K_CANDIDATES` (=40) nearest neighbors by
combined-embedding cosine similarity — run separately against S2 and S3. That's the entire
retrieval strategy: no exact-name channel, no global (cross-country) fallback. This is a
deliberate simplification versus the earlier version:

- It keeps the pipeline to exactly one retrieval signal (`faiss_score` / `retrieval_rank`),
  which is what Step 6's feature set is built around.
- If an S1 record's country string genuinely has zero matching records in S2/S3 (e.g. a data
  gap, not just a naming mismatch — country normalization in Step 1 already reconciles
  spelling/case/abbreviation differences), it will surface zero candidates here and end up as
  an explicit *no-match* row in the final submission via Step 10's completeness-fill step —
  which is the correct outcome, not a bug, since there is nothing on the candidate side to
  match it to.
- **Before trusting this**, Step 7 measures candidate-generation recall against the ground
  truth. If recall comes in low, the fix is to raise `TOP_K_CANDIDATES` or re-check that
  `country_norm` values line up between S1 and S2/S3 — not to add retrieval channels back in.

Every query batch's FAISS search runs **on GPU** in chunks of `FAISS_QUERY_BATCH` S1 rows (so
peak GPU memory for search is bounded regardless of how large a country block's S1 side is),
and results for each country are **flushed to a CSV on disk as soon as they're computed** —
they are never accumulated into one big in-RAM list across the whole S1 table.

In [ ]:
def run_retrieval_to_disk(s1_df, s1_emb, country_indices, top_k, candidate_source, out_csv_path,
                           query_batch=FAISS_QUERY_BATCH):
    # Country-blocked FAISS top-k retrieval, batched on GPU, streamed to disk.
    #
    # For every S1 country group: look up that country's candidate-side FAISS index and search
    # it for the top `top_k` neighbors. Query vectors are read from the on-disk embedding memmap
    # in `query_batch`-sized chunks (never the whole S1 pool at once, and upcast
    # float16->float32 only for that chunk); every country's results are appended to
    # `out_csv_path` immediately after that country finishes, so peak RAM is bounded to one
    # country's worth of candidate rows, not the full candidate table.
    out_path = Path(out_csv_path)
    if out_path.exists():
        out_path.unlink()
    header_written = False
    out_cols = ["source1_entity_id", "candidate_entity_id", "candidate_source",
                "faiss_score", "retrieval_rank"]

    for country, group in s1_df.groupby("country_norm", observed=True):
        if country not in country_indices:
            continue  # no candidate-side records at all for this country -> zero candidates
        index_obj, cand_ids = country_indices[country]
        k = min(top_k, index_obj.ntotal)
        if k == 0:
            continue

        row_positions = group.index.values
        s1_ids = group["entity_id"].values
        rows_buffer = []

        for start in range(0, len(row_positions), query_batch):
            chunk_pos = row_positions[start:start + query_batch]
            query_vecs = np.asarray(s1_emb[chunk_pos], dtype="float32")
            sims, neighbors = index_obj.search(query_vecs, k)  # GPU search if index is GPU-resident

            for local_i in range(len(chunk_pos)):
                s1_id = s1_ids[start + local_i]
                for rank in range(k):
                    cand_pos = neighbors[local_i, rank]
                    if cand_pos == -1:
                        continue
                    rows_buffer.append((s1_id, cand_ids[cand_pos], candidate_source,
                                         float(sims[local_i, rank]), rank + 1))

        if rows_buffer:
            chunk_df = pd.DataFrame(rows_buffer, columns=out_cols)
            chunk_df.to_csv(out_path, mode="a", header=not header_written, index=False)
            header_written = True
        del rows_buffer

    if not header_written:
        pd.DataFrame(columns=out_cols).to_csv(out_path, index=False)
    print(f"Wrote {candidate_source} candidate rows to {out_path}")

In [ ]:
s1_train = s1_train.reset_index(drop=True)

RAW_CAND_TRAIN_S2_PATH = f"{CAND_DIR}/raw_candidates_train_s2.csv"
RAW_CAND_TRAIN_S3_PATH = f"{CAND_DIR}/raw_candidates_train_s3.csv"

run_retrieval_to_disk(s1_train, s1_train_emb, s2_country_idx, TOP_K_CANDIDATES, "S2", RAW_CAND_TRAIN_S2_PATH)
run_retrieval_to_disk(s1_train, s1_train_emb, s3_country_idx, TOP_K_CANDIDATES, "S3", RAW_CAND_TRAIN_S3_PATH)

# ----- Reload from disk rather than reusing anything held in RAM from the retrieval loop -----
raw_candidates_train = pd.concat(
    [pd.read_csv(RAW_CAND_TRAIN_S2_PATH), pd.read_csv(RAW_CAND_TRAIN_S3_PATH)],
    ignore_index=True,
)
print(f"Raw (pre-dedup) candidate rows, reloaded from disk: {len(raw_candidates_train):,}")
display(raw_candidates_train.head())

## Step 5 — Dedup + Disk Checkpoint

With a single retrieval channel, duplicates should only happen if the notebook is re-run
without clearing old output — `dedupe_candidates` collapses to one row per
`(source1_entity_id, candidate_entity_id, candidate_source)`, keeping the max FAISS score and
best (lowest) rank seen for that pair. The deduped candidate table is then **written to disk
and reloaded** before feature engineering (Step 6) begins, so the modelling stage always reads
its input from disk rather than continuing to hold the in-memory object built during retrieval.

In [ ]:
def dedupe_candidates(raw_df):
    # Collapse raw candidate rows into one row per (S1, candidate, source) pair.
    key_cols = ["source1_entity_id", "candidate_entity_id", "candidate_source"]
    agg = (raw_df.groupby(key_cols)
           .agg(faiss_score=("faiss_score", "max"), retrieval_rank=("retrieval_rank", "min"))
           .reset_index())
    return agg

In [ ]:
CAND_PAIRS_TRAIN_PATH = f"{CAND_DIR}/candidate_pairs_train.csv"

candidate_pairs_train = dedupe_candidates(raw_candidates_train)
candidate_pairs_train.to_csv(CAND_PAIRS_TRAIN_PATH, index=False)

# Drop the in-RAM objects and reload strictly from disk before modelling.
del raw_candidates_train, candidate_pairs_train
gc.collect()
candidate_pairs_train = pd.read_csv(CAND_PAIRS_TRAIN_PATH)

print(f"Final candidate_pairs (train, reloaded from disk): {len(candidate_pairs_train):,} unique "
      f"(S1, candidate, source) pairs")
print(f"  covering {candidate_pairs_train['source1_entity_id'].nunique():,} / {len(s1_train):,} S1 records")
display(candidate_pairs_train.head())

## Step 6 — Feature Engineering: RapidFuzz + Jaccard + FAISS score/rank

For every candidate pair, join in the S1 record and the S2/S3 candidate record, then compute:

- **Retrieval features** — `faiss_score` (combined-embedding cosine similarity) and
  `retrieval_rank` (1 = closest neighbor in its FAISS search) — the only two signals coming out
  of Step 4
- **Normalization features** — exact-match booleans on each normalized name variant, address
  exact match, country match
- **RapidFuzz features** — `ratio`, `token_sort_ratio`, `token_set_ratio` for name and address
- **Jaccard / token-overlap features** — Jaccard similarity of name/address token sets, Jaccard
  overlap of numeric tokens (catches PIN-code / house-number agreement even when the rest of
  the address differs), plus simple length/count-diff signals

The RapidFuzz and Jaccard passes are computed with plain Python `zip()` loops over `.tolist()`
values rather than `DataFrame.apply(axis=1)` — `apply` re-enters the Python/pandas row
machinery on every call, while a `zip()` loop calls straight into RapidFuzz's C implementation
with negligible overhead, which matters once the candidate table reaches millions of rows.

In [ ]:
FIELDS_FOR_MERGE = [
    "entity_id", "name_norm", "name_tokens", "name_suffix_norm", "name_nosuffix", "name_sorted",
    "address_norm", "address_tokens", "address_numeric_tokens", "country_norm",
]


def jaccard(a, b):
    a, b = set(a or []), set(b or [])
    if not a and not b:
        return 0.0
    union = a | b
    return len(a & b) / len(union) if union else 0.0


def attach_pair_context(pairs_df, s1_df, s2_df, s3_df):
    # Merge S1 fields and candidate (S2/S3) fields onto each candidate pair row.
    cand_lookup_df = pd.concat([s2_df[FIELDS_FOR_MERGE], s3_df[FIELDS_FOR_MERGE]], ignore_index=True)

    s1_sub = s1_df[FIELDS_FOR_MERGE].add_prefix("s1_").rename(columns={"s1_entity_id": "source1_entity_id"})
    cand_sub = cand_lookup_df.add_prefix("cand_").rename(columns={"cand_entity_id": "candidate_entity_id"})

    merged = (pairs_df
              .merge(s1_sub, on="source1_entity_id", how="left")
              .merge(cand_sub, on="candidate_entity_id", how="left"))
    return merged


def compute_features(merged_df):
    # Compute the full feature set described above. Returns a new dataframe with feature columns added.
    df = merged_df.copy()

    # --- RapidFuzz string-similarity features (zip-loop, not .apply; rescaled 0-100 -> 0-1) ---
    s1_name = df["s1_name_norm"].fillna("").tolist()
    cand_name = df["cand_name_norm"].fillna("").tolist()
    s1_addr = df["s1_address_norm"].fillna("").tolist()
    cand_addr = df["cand_address_norm"].fillna("").tolist()

    df["name_ratio"] = [fuzz.ratio(a, b) / 100.0 for a, b in zip(s1_name, cand_name)]
    df["name_token_sort_ratio"] = [fuzz.token_sort_ratio(a, b) / 100.0 for a, b in zip(s1_name, cand_name)]
    df["name_token_set_ratio"] = [fuzz.token_set_ratio(a, b) / 100.0 for a, b in zip(s1_name, cand_name)]
    df["address_ratio"] = [fuzz.ratio(a, b) / 100.0 for a, b in zip(s1_addr, cand_addr)]
    df["address_token_sort_ratio"] = [fuzz.token_sort_ratio(a, b) / 100.0 for a, b in zip(s1_addr, cand_addr)]
    df["address_token_set_ratio"] = [fuzz.token_set_ratio(a, b) / 100.0 for a, b in zip(s1_addr, cand_addr)]

    # --- Normalization / exact-match features ---
    df["name_exact_match"] = (df["s1_name_norm"] == df["cand_name_norm"]).astype(int)
    df["name_suffix_exact_match"] = (df["s1_name_suffix_norm"] == df["cand_name_suffix_norm"]).astype(int)
    df["name_nosuffix_exact_match"] = (df["s1_name_nosuffix"] == df["cand_name_nosuffix"]).astype(int)
    df["name_sorted_exact_match"] = (df["s1_name_sorted"] == df["cand_name_sorted"]).astype(int)
    df["address_exact_match"] = (df["s1_address_norm"] == df["cand_address_norm"]).astype(int)
    df["country_match"] = (df["s1_country_norm"].astype(str) == df["cand_country_norm"].astype(str)).astype(int)

    # --- Jaccard / token-overlap features (zip-loop) ---
    df["name_token_jaccard"] = [jaccard(a, b) for a, b in zip(df["s1_name_tokens"], df["cand_name_tokens"])]
    df["address_token_jaccard"] = [jaccard(a, b) for a, b in zip(df["s1_address_tokens"], df["cand_address_tokens"])]
    df["numeric_token_overlap"] = [jaccard(a, b) for a, b in
                                    zip(df["s1_address_numeric_tokens"], df["cand_address_numeric_tokens"])]
    df["name_token_count_diff"] = [abs(len(a or []) - len(b or []))
                                    for a, b in zip(df["s1_name_tokens"], df["cand_name_tokens"])]
    df["address_token_count_diff"] = [abs(len(a or []) - len(b or []))
                                       for a, b in zip(df["s1_address_tokens"], df["cand_address_tokens"])]
    df["name_length_diff"] = (df["s1_name_norm"].str.len() - df["cand_name_norm"].str.len()).abs()

    # --- Retrieval features: defensive fill in case of an unexpected merge gap ---
    df["faiss_score"] = df["faiss_score"].fillna(0.0)
    df["retrieval_rank"] = df["retrieval_rank"].fillna(TOP_K_CANDIDATES + 1)

    return df


FEATURE_COLUMNS = [
    # Retrieval features
    "faiss_score", "retrieval_rank",
    # Normalization features
    "name_exact_match", "name_suffix_exact_match", "name_nosuffix_exact_match",
    "name_sorted_exact_match", "address_exact_match", "country_match",
    # RapidFuzz features
    "name_ratio", "name_token_sort_ratio", "name_token_set_ratio",
    "address_ratio", "address_token_sort_ratio", "address_token_set_ratio",
    # Jaccard / token-overlap features
    "name_token_jaccard", "address_token_jaccard", "numeric_token_overlap",
    "name_token_count_diff", "address_token_count_diff", "name_length_diff",
]


def build_feature_matrix(pairs_df, s1_df, s2_df, s3_df):
    # End-to-end: attach context + compute features. Used for both train and test.
    merged = attach_pair_context(pairs_df, s1_df, s2_df, s3_df)
    features = compute_features(merged)
    for col in FEATURE_COLUMNS:
        assert col in features.columns, f"Missing expected feature column: {col}"
    return features

In [ ]:
train_features = build_feature_matrix(candidate_pairs_train, s1_train, s2_train, s3_train)
print(f"Feature matrix (train): {train_features.shape}")
display(train_features[["source1_entity_id", "candidate_entity_id", "candidate_source"] + FEATURE_COLUMNS].head())

## Step 7 — Label Creation & Label-Distribution Check

1. Parse `train_ground_truth.tsv` into `{source1_entity_id: {matched candidate ids}}`
2. Label every candidate pair `1` if it's a true match, else `0` (a plain set-membership check,
   O(1) per pair)
3. **Candidate-generation recall check**: the fraction of ground-truth pairs that actually made
   it into our (now much simpler) candidate set. If this isn't close to 1.0, `TOP_K_CANDIDATES`
   in Step 4 needs raising *before* the model can possibly do well — it can only rank the
   candidates it's given.
4. **Label distribution check**: counts, overall positive rate, and positive rate broken out by
   candidate source (S2 vs. S3) and by S1 country — a heavily imbalanced or oddly skewed
   distribution here is worth catching before training, not after.

In [ ]:
def parse_ground_truth(gt_df):
    # Return {source1_entity_id: set(matched_entity_ids)}. Missing/blank -> empty set.
    gt_map = {}
    for row in gt_df.itertuples(index=False):
        matched = row.matched_entity_ids
        if pd.isna(matched) or str(matched).strip() == "":
            gt_map[row.source1_entity_id] = set()
        else:
            gt_map[row.source1_entity_id] = {x.strip() for x in str(matched).split(",") if x.strip()}
    return gt_map


def label_candidate_pairs(features_df, gt_map):
    # Set-based membership check: O(1) per pair instead of a dict.get() inside .apply().
    gt_pairs = {(s1, c) for s1, cands in gt_map.items() for c in cands}
    features_df = features_df.copy()
    features_df["label"] = [
        int((s1, c) in gt_pairs)
        for s1, c in zip(features_df["source1_entity_id"], features_df["candidate_entity_id"])
    ]
    return features_df


def report_candidate_recall(features_df, gt_map):
    # Diagnostic: what fraction of ground-truth pairs survived candidate generation?
    total_gt_pairs = sum(len(v) for v in gt_map.values())
    found_pairs = features_df["label"].sum()
    recall = found_pairs / total_gt_pairs if total_gt_pairs > 0 else float("nan")
    print(f"Ground-truth pairs: {total_gt_pairs:,} | found in candidate set: {found_pairs:,} "
          f"| candidate-generation recall: {recall:.4f}")
    if recall < 0.98:
        print("  WARNING: recall below 0.98 -- consider raising TOP_K_CANDIDATES in Step 4, "
              "or double-check country_norm alignment between S1 and S2/S3, before trusting the model.")
    return recall

In [ ]:
gt_map_train = parse_ground_truth(gt_train)
train_features = label_candidate_pairs(train_features, gt_map_train)
_ = report_candidate_recall(train_features, gt_map_train)

print(f"\nOverall positive rate in candidate pairs: {train_features['label'].mean():.4%}")

In [ ]:
# ----- Label distribution check -----
print("Label counts:")
print(train_features["label"].value_counts())
print("\nLabel rate:")
print(train_features["label"].value_counts(normalize=True))
print("\nPositive rate by candidate_source:")
print(train_features.groupby("candidate_source")["label"].mean())
print("\nPositive rate by S1 country:")
print(train_features.groupby("s1_country_norm", observed=True)["label"].mean())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

train_features["label"].value_counts().sort_index().plot(
    kind="bar", ax=axes[0], color=["#c0504d", "#4f81bd"])
axes[0].set_xticklabels(["no_match (0)", "match (1)"], rotation=0)
axes[0].set_title("Label distribution -- candidate pairs")
axes[0].set_ylabel("count")

train_features.groupby("candidate_source")["label"].mean().plot(kind="bar", ax=axes[1], color="#4f81bd")
axes[1].set_title("Positive rate by candidate_source")
axes[1].set_ylabel("mean(label)")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## Step 8 — Group-Based Split & LightGBM Training

Group-based train/validation split on `source1_entity_id`, so no S1 entity's candidates leak
across the split, then a LightGBM binary classifier with imbalance handling
(`is_unbalance=True`) and early stopping on the validation set.

In [ ]:
# Group-based split -- every S1 entity's candidates stay entirely in train OR validation.
gss = GroupShuffleSplit(n_splits=1, test_size=VALIDATION_FRACTION, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss.split(train_features, groups=train_features["source1_entity_id"]))

train_split = train_features.iloc[train_idx].reset_index(drop=True)
val_split = train_features.iloc[val_idx].reset_index(drop=True)

assert set(train_split["source1_entity_id"]).isdisjoint(set(val_split["source1_entity_id"])), \
    "Data leakage: some S1 entities appear in both train and validation!"

print(f"Train: {len(train_split):,} pairs / {train_split['source1_entity_id'].nunique():,} S1 entities "
      f"(pos rate {train_split['label'].mean():.4%})")
print(f"Val:   {len(val_split):,} pairs / {val_split['source1_entity_id'].nunique():,} S1 entities "
      f"(pos rate {val_split['label'].mean():.4%})")

In [ ]:
X_train, y_train = train_split[FEATURE_COLUMNS].astype(float), train_split["label"]
X_val, y_val = val_split[FEATURE_COLUMNS].astype(float), val_split["label"]

lgb_train_set = lgb.Dataset(X_train, label=y_train)
lgb_val_set = lgb.Dataset(X_val, label=y_val, reference=lgb_train_set)

model = lgb.train(
    LGBM_PARAMS,
    lgb_train_set,
    num_boost_round=LGBM_NUM_BOOST_ROUND,
    valid_sets=[lgb_train_set, lgb_val_set],
    valid_names=["train", "val"],
    callbacks=[
        lgb.early_stopping(LGBM_EARLY_STOPPING_ROUNDS),
        lgb.log_evaluation(period=50),
    ],
)

In [ ]:
# Feature importance -- sanity check that the model is leaning on sensible signals.
importance_df = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "importance": model.feature_importance(importance_type="gain"),
}).sort_values("importance", ascending=False)
display(importance_df)

## Step 9 — Threshold Tuning for F0.5

F0.5 weights precision more heavily than recall (appropriate here since over-matching pollutes
the submission with false positives). We sweep the decision threshold to maximize
**F0.5 on the match class** (`average="binary"`, the standard target for a matching task —
optimizing "macro" F0.5 instead would let the much larger, near-trivially-easy no-match class
dominate the score and mask how well the model actually finds matches). Macro F0.5 is reported
alongside purely for reference.

In [ ]:
def optimize_threshold_f05(y_true, y_proba, thresholds=None):
    # Sweep thresholds, maximizing F0.5 on the positive (match) class.
    if thresholds is None:
        thresholds = np.arange(0.01, 1.00, 0.01)
    best_threshold, best_score = 0.5, -1.0
    for t in thresholds:
        preds = (y_proba >= t).astype(int)
        score = fbeta_score(y_true, preds, beta=0.5, average="binary", zero_division=0)
        if score > best_score:
            best_score, best_threshold = score, t
    return best_threshold, best_score


val_proba = model.predict(X_val, num_iteration=model.best_iteration)
best_threshold, best_f05 = optimize_threshold_f05(y_val.values, val_proba)
print(f"Best threshold: {best_threshold:.2f}  |  F0.5 (match class): {best_f05:.4f}")

val_preds = (val_proba >= best_threshold).astype(int)
macro_f05 = fbeta_score(y_val, val_preds, beta=0.5, average="macro", zero_division=0)
print(f"F0.5 (macro, for reference): {macro_f05:.4f}")
print(f"\nPrecision: {precision_score(y_val, val_preds, zero_division=0):.4f}  "
      f"Recall: {recall_score(y_val, val_preds, zero_division=0):.4f}")
print(classification_report(y_val, val_preds, target_names=["no_match", "match"], zero_division=0))

In [ ]:
# Persist the model + threshold + feature list so Step 10 can be re-run independently later.
joblib.dump(
    {"model": model, "threshold": best_threshold, "feature_columns": FEATURE_COLUMNS},
    f"{MODEL_DIR}/entity_matcher.joblib",
)
print(f"Saved model artifact to {MODEL_DIR}/entity_matcher.joblib")

## Step 10 — Inference, `candidates.csv`, `matching_pairs.csv` & Submission

Everything below **reuses the exact same functions from Steps 1–6** on the test files — no
logic is duplicated. This matters most for country handling: France appears only in the test
set, but since blocking/indexing/retrieval never hard-code a country list, France's records are
grouped, indexed, and searched exactly like any other country automatically.

Three files come out of this step:

1. **`candidates.csv`** — every candidate pair that survived country-block + top-40 FAISS
   retrieval for the test set, with every engineered feature *and* the model's predicted
   `match_proba`. This is the full, auditable candidate table before any thresholding.
2. **`matching_pairs.csv`** — just the pairs the model predicts as matches
   (`match_proba >= best_threshold`), with no cap on how many per source and no completeness
   padding — i.e. the model's raw matching decisions.
3. **`submission.tsv`** — the official format: `matching_pairs.csv` capped to the top
   `TOP_K_PER_SOURCE_IN_SUBMISSION` per (S1, source), with every test S1 entity guaranteed to
   appear (even those with zero predicted matches) via an explicit completeness-fill step.

In [ ]:
# ----- Load & normalize test data (reuses load_source_file from Step 1) -----
s1_test = load_source_file(TEST_S1_PATH).reset_index(drop=True)
s2_test = load_source_file(TEST_S2_PATH).reset_index(drop=True)
s3_test = load_source_file(TEST_S3_PATH).reset_index(drop=True)

print(f"S1 test: {len(s1_test):,} | S2 test: {len(s2_test):,} | S3 test: {len(s3_test):,}")
print("Countries in test data:",
      sorted(set(s1_test['country_norm']) | set(s2_test['country_norm']) | set(s3_test['country_norm'])))

In [ ]:
# ----- Embeddings (reuses Step 2 helpers) -----
s1_test_emb = get_or_compute_combined_embeddings(s1_test, "test_s1", embedding_model)
s2_test_emb = get_or_compute_combined_embeddings(s2_test, "test_s2", embedding_model)
s3_test_emb = get_or_compute_combined_embeddings(s3_test, "test_s3", embedding_model)

In [ ]:
# ----- Country-wise FAISS indices (reuses Step 3 helper) -----
s2_country_idx_test = build_country_indices(s2_test, s2_test_emb)
s3_country_idx_test = build_country_indices(s3_test, s3_test_emb)

print("S2 test countries indexed:", sorted(s2_country_idx_test.keys()))
print("S3 test countries indexed:", sorted(s3_country_idx_test.keys()))

In [ ]:
# ----- Candidate retrieval, streamed to disk (reuses Step 4 helper) -----
RAW_CAND_TEST_S2_PATH = f"{CAND_DIR}/raw_candidates_test_s2.csv"
RAW_CAND_TEST_S3_PATH = f"{CAND_DIR}/raw_candidates_test_s3.csv"

run_retrieval_to_disk(s1_test, s1_test_emb, s2_country_idx_test, TOP_K_CANDIDATES, "S2", RAW_CAND_TEST_S2_PATH)
run_retrieval_to_disk(s1_test, s1_test_emb, s3_country_idx_test, TOP_K_CANDIDATES, "S3", RAW_CAND_TEST_S3_PATH)

raw_candidates_test = pd.concat(
    [pd.read_csv(RAW_CAND_TEST_S2_PATH), pd.read_csv(RAW_CAND_TEST_S3_PATH)],
    ignore_index=True,
)
print(f"Raw test candidates, reloaded from disk: {len(raw_candidates_test):,}")

In [ ]:
# ----- Dedup + disk checkpoint + feature engineering (reuses Step 5-6 helpers) -----
CAND_PAIRS_TEST_PATH = f"{CAND_DIR}/candidate_pairs_test.csv"

candidate_pairs_test = dedupe_candidates(raw_candidates_test)
candidate_pairs_test.to_csv(CAND_PAIRS_TEST_PATH, index=False)
del raw_candidates_test, candidate_pairs_test
gc.collect()
candidate_pairs_test = pd.read_csv(CAND_PAIRS_TEST_PATH)

test_features = build_feature_matrix(candidate_pairs_test, s1_test, s2_test, s3_test)
print(f"Test feature matrix: {test_features.shape}, "
      f"covering {test_features['source1_entity_id'].nunique():,} / {len(s1_test):,} S1 test entities")

In [ ]:
# ----- Predict match probability -----
X_test = test_features[FEATURE_COLUMNS].astype(float)
test_features["match_proba"] = model.predict(X_test, num_iteration=model.best_iteration)

# ----- Deliverable 1: candidates.csv -- every surviving candidate + features + match_proba -----
candidates_cols = ["source1_entity_id", "candidate_entity_id", "candidate_source"] + FEATURE_COLUMNS + ["match_proba"]
test_features[candidates_cols].sort_values(
    ["source1_entity_id", "candidate_source", "match_proba"], ascending=[True, True, False]
).to_csv(CANDIDATES_CSV_PATH, index=False)
print(f"Saved {len(test_features):,} candidate rows to {CANDIDATES_CSV_PATH}")

# ----- Deliverable 2: matching_pairs.csv -- only pairs at/above the tuned threshold -----
matching_pairs = (test_features[test_features["match_proba"] >= best_threshold]
                   [["source1_entity_id", "candidate_entity_id", "candidate_source", "match_proba"]]
                   .sort_values(["source1_entity_id", "candidate_source", "match_proba"],
                                ascending=[True, True, False])
                   .reset_index(drop=True))
matching_pairs.to_csv(MATCHING_PAIRS_CSV_PATH, index=False)
print(f"Saved {len(matching_pairs):,} predicted matching pairs to {MATCHING_PAIRS_CSV_PATH} "
      f"({matching_pairs['source1_entity_id'].nunique():,} S1 entities with >=1 predicted match)")

In [ ]:
def build_submission(features_df, threshold, top_k_per_source=TOP_K_PER_SOURCE_IN_SUBMISSION):
    # Keep pairs at/above `threshold`, then per (S1, candidate_source) keep only the top-K by
    # predicted probability. Returns columns: source1_entity_id, candidate_entity_id, score,
    # candidate_source.
    df = features_df.copy()
    df["is_match_pred"] = (df["match_proba"] >= threshold).astype(int)
    matched = df[df["is_match_pred"] == 1].copy()

    matched = matched.sort_values(
        ["source1_entity_id", "candidate_source", "match_proba"], ascending=[True, True, False]
    )
    matched["rank_within_source"] = matched.groupby(
        ["source1_entity_id", "candidate_source"]
    ).cumcount() + 1
    top_matches = matched[matched["rank_within_source"] <= top_k_per_source]

    return (top_matches[["source1_entity_id", "candidate_entity_id", "match_proba", "candidate_source"]]
            .rename(columns={"match_proba": "score"})
            .reset_index(drop=True))


submission = build_submission(test_features, best_threshold)
print(f"Submission rows before completeness fill: {len(submission):,} "
      f"({submission['source1_entity_id'].nunique():,} S1 entities with >=1 predicted match)")

In [ ]:
# ----- Ensure every test S1 entity appears, even with zero predicted matches -----
all_test_s1_ids = s1_test["entity_id"].tolist()
covered_ids = set(submission["source1_entity_id"])
missing_ids = set(all_test_s1_ids) - covered_ids

if missing_ids:
    filler = pd.DataFrame({
        "source1_entity_id": sorted(missing_ids),
        "candidate_entity_id": "",   # no predicted match for this S1 entity
        "score": 0.0,
        "candidate_source": "",
    })
    # NOTE: adjust this block if the official submission format instead expects these S1 entities
    # to simply be omitted rather than represented with an empty candidate row.
    submission = pd.concat([submission, filler], ignore_index=True)

submission = submission.sort_values(["source1_entity_id", "candidate_source", "score"],
                                     ascending=[True, True, False]).reset_index(drop=True)

# ----- Final sanity checks -----
assert submission["source1_entity_id"].nunique() == len(all_test_s1_ids), \
    "Not every test S1 entity is represented in the submission!"
assert not submission.duplicated(subset=["source1_entity_id", "candidate_entity_id", "candidate_source"]).any(), \
    "Duplicate (S1, candidate, source) rows in submission!"

print(f"Final submission: {len(submission):,} rows covering all {len(all_test_s1_ids):,} test S1 entities")
display(submission.head(10))

In [ ]:
submission.to_csv(SUBMISSION_PATH, sep="\t", index=False)
print(f"Saved submission to {SUBMISSION_PATH}")
print(f"Saved candidates to {CANDIDATES_CSV_PATH}")
print(f"Saved matching pairs to {MATCHING_PAIRS_CSV_PATH}")

## Notes & Possible Improvements

- **Scale**: `IndexFlatIP` is exact but O(n) per query. For very large S2/S3 pools per country,
  swap in an approximate index (`IndexIVFFlat` / `IndexHNSWFlat`) for large countries while
  keeping `IndexFlatIP` for small ones — check block sizes first; at Kaggle/EC2 scale a flat
  index is usually still fast enough on GPU.
- **Recall safety net**: candidate generation here is deliberately just country-block + top-40
  FAISS, no fallback. If Step 7's recall check comes in low specifically for records whose
  country label doesn't perfectly line up between S1 and S2/S3 (as opposed to genuinely absent
  countries), a targeted global-fallback search (only for S1 rows whose country has zero
  candidate-side index) is the smallest fix to add back.
- **Transliteration**: for Indian/French addresses and names with regional-script or accented
  variants, consider a transliteration-normalization step before Step 1's tokenization, and
  double check the embedding model's multilingual coverage on your actual data.
- **Model**: a LightGBM *ranker* (`lambdarank`) grouped by `source1_entity_id` is worth
  comparing against the current binary classifier, since the final task is inherently a
  per-S1 ranking/selection problem.
- **Calibration**: if `match_proba` needs to be interpretable as a true probability (rather than
  just a ranking score), apply `sklearn.calibration.CalibratedClassifierCV` on top of the
  LightGBM output.
- **Submission format**: double-check whether S1 entities with zero matches should appear as an
  empty-candidate row (current behavior) or be omitted entirely — adjust the "completeness fill"
  cell in Step 10 accordingly once you have the official format spec.